# Notebook 01 — Data Exploration
**Weather-Aware Flood Segmentation | Sen1Floods11 Dataset**

This notebook covers:
1. Dataset structure and file counts
2. Visualising Sentinel-1 SAR images and flood masks
3. Class balance analysis (flood vs non-flood pixels)
4. Weather feature distribution
5. Inter-event statistics

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────────────────
import sys
os.chdir(r'd:\flood_segmentation')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/flood_segmentation
    %pip install -q rasterio tqdm scikit-learn
print('Environment ready.')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import rasterio
from rasterio.enums import Resampling

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120})

# Paths
DATA_DIR     = 'data/raw'
S1_DIR       = os.path.join(DATA_DIR, 'S1Hand')
LABEL_DIR    = os.path.join(DATA_DIR, 'S1HandLabels')
WEATHER_CSV  = 'data/processed/weather.csv'
FIGURES_DIR  = 'results/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f'S1Hand images    : {len(os.listdir(S1_DIR))}')
print(f'Label masks      : {len(os.listdir(LABEL_DIR))}')

In [ ]:
# --- 1. File inventory by flood event ---
files   = sorted(f for f in os.listdir(S1_DIR) if f.endswith('.tif'))
events  = {}
for f in files:
    ev = f.split('_')[0]
    events[ev] = events.get(ev, 0) + 1

ev_df = pd.Series(events, name='count').sort_values(ascending=False)
print('\nFiles per flood event:')
print(ev_df.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
ev_df.plot(kind='bar', ax=ax, color=sns.color_palette('Blues_r', len(ev_df)), edgecolor='white')
ax.set(title='Sen1Floods11 — Chips per Flood Event', xlabel='Event', ylabel='# Chips')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '01_chips_per_event.png'), dpi=150)
plt.show()

In [ ]:
# --- 2. Visualise sample images and masks ---
def load_tif(path, size=256):
    with rasterio.open(path) as src:
        return src.read(out_shape=(src.count, size, size),
                        resampling=Resampling.bilinear).astype(np.float32)

def norm_sar(img):
    out = np.zeros_like(img)
    for c in range(img.shape[0]):
        p2, p98 = np.percentile(img[c], (2, 98))
        out[c] = np.clip((img[c] - p2) / (p98 - p2 + 1e-8), 0, 1)
    return out

# Sample one chip per event (up to 6 events)
sample_events = list(events.keys())[:6]
fig, axes = plt.subplots(len(sample_events), 3, figsize=(12, 4 * len(sample_events)))
col_titles = ['VV (dB)', 'VH (dB)', 'Flood Mask']
for col, ct in enumerate(col_titles):
    axes[0][col].set_title(ct, fontsize=13, fontweight='bold')

for row, ev in enumerate(sample_events):
    ev_files = [f for f in files if f.startswith(ev)]
    fname    = ev_files[len(ev_files) // 2]   # pick middle chip
    img  = norm_sar(load_tif(os.path.join(S1_DIR, fname)))
    lbl  = load_tif(os.path.join(LABEL_DIR, fname.replace('S1Hand', 'LabelHand')))[0]
    lbl  = np.clip(lbl, 0, 1)

    axes[row][0].imshow(img[0], cmap='gray', vmin=0, vmax=1)
    axes[row][1].imshow(img[1], cmap='gray', vmin=0, vmax=1)
    axes[row][2].imshow(lbl,   cmap='Blues', vmin=0, vmax=1)
    axes[row][0].set_ylabel(ev, fontsize=11, rotation=90, labelpad=6)
    for ax in axes[row]: ax.axis('off')

plt.suptitle('Sample Sentinel-1 Chips and Flood Masks', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '01_sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 3. Class balance analysis ---
print('Sampling flood pixel ratios (this may take a minute)...')
ratios = []
sample_n = min(200, len(files))
sampled  = np.random.choice(files, sample_n, replace=False)
for fname in sampled:
    lbl = load_tif(os.path.join(LABEL_DIR, fname.replace('S1Hand', 'LabelHand')), size=128)[0]
    lbl = np.clip(lbl, 0, 1)
    ratios.append(lbl.mean())

ratios = np.array(ratios)
print(f'Mean flood ratio : {ratios.mean():.3f}  ({ratios.mean()*100:.1f}% flood pixels per chip)')
print(f'Median           : {np.median(ratios):.3f}')
print(f'Chips with <5%%  : {(ratios < 0.05).sum()}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ratios, bins=30, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(ratios.mean(), color='#e74c3c', lw=2, label=f'Mean = {ratios.mean():.3f}')
ax.set(title='Flood Pixel Ratio per Chip', xlabel='Fraction of flood pixels', ylabel='# Chips')
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '01_class_balance.png'), dpi=150)
plt.show()
print('Class imbalance confirmed → BCE+Dice loss is important!')

In [ ]:
# --- 4. Weather feature distributions ---
weather_df = pd.read_csv(WEATHER_CSV)
print(weather_df.describe().round(2))

features = ['precipitation', 'temperature', 'humidity', 'wind_speed', 'pressure']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
colors = sns.color_palette('husl', 5)
for ax, feat, col in zip(axes, features, colors):
    ax.hist(weather_df[feat], bins=25, color=col, edgecolor='white', alpha=0.85)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('Value')
plt.suptitle('Weather Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '01_weather_distributions.png'), dpi=150)
plt.show()

In [ ]:
# --- 5. Weather correlation matrix ---
fig, ax = plt.subplots(figsize=(7, 6))
corr = weather_df[features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, square=True)
ax.set_title('Weather Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '01_weather_correlation.png'), dpi=150)
plt.show()
print('Data exploration complete. Proceed to Notebook 02.')